In [1]:
import pandas as pd

## Loading data

In [2]:
train = pd.read_csv(r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\train.csv\train.csv", parse_dates=["date"])
stores = pd.read_csv(r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\stores.csv\stores.csv")
items = pd.read_csv(r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\items.csv\items.csv")
oil = pd.read_csv(r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\oil.csv\oil.csv", parse_dates=["date"])
holidays = pd.read_csv(r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\holidays_events.csv\holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\transactions.csv\transactions.csv", parse_dates=["date"])

C:\Users\argon\AppData\Local\Temp\ipykernel_15760\1184044644.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\train.csv\train.csv", parse_dates=["date"])


In [4]:
train.tail()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
125497035,125497035,2017-08-15,54,2089339,4.0,False
125497036,125497036,2017-08-15,54,2106464,1.0,True
125497037,125497037,2017-08-15,54,2110456,192.0,False
125497038,125497038,2017-08-15,54,2113914,198.0,True
125497039,125497039,2017-08-15,54,2116416,2.0,False


## Choose stores (volume-based)

In [5]:
store_volume = (
    train.groupby("store_nbr")["unit_sales"]
    .sum()
    .sort_values(ascending=False)
)

sample_stores = store_volume.head(25).index.tolist()

## Choose stores (item-based)

In [6]:
item_history = (
    train.groupby("item_nbr")
    .agg(
        total_sales=("unit_sales", "sum"),
        n_obs=("unit_sales", "count")
    )
)

sample_items = (
    item_history
    .query("n_obs > 500")                 # ensure sufficient history
    .sort_values("total_sales", ascending=False)
    .head(800)                            # broadened but controlled
    .index
    .tolist()
)

## Filter training data

In [7]:
df = train[
    train["store_nbr"].isin(sample_stores) &
    train["item_nbr"].isin(sample_items)
].copy()


## Restricting time window

In [8]:
df = df[
    (df["date"] >= "2013-01-01") &
    (df["date"] <= "2015-12-31")
]

In [9]:
df.shape

(14218932, 6)

## Merge auxiliary tables

In [10]:
df = df.merge(items, on="item_nbr", how="left")
df = df.merge(stores, on="store_nbr", how="left")
df = df.merge(oil, on="date", how="left")
df = df.merge(
    holidays[["date", "description"]],
    on="date",
    how="left"
)
df = df.merge(
    transactions,
    on=["date", "store_nbr"],
    how="left"
)


## Size check

In [11]:
df.shape
df.memory_usage(deep=True).sum() / 1e9


5.802867243

## Sanity check

In [14]:
df.columns

Index(['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'onpromotion',
       'family', 'class', 'perishable', 'city', 'state', 'type', 'cluster',
       'dcoilwtico', 'description', 'transactions'],
      dtype='object')

## Safety Prep

In [12]:
import pandas as pd
import numpy as np

df = df.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)

## Calendar features (cheap, high signal)

In [13]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["dayofweek"] = df["date"].dt.dayofweek
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)


## Holiday signal

In [14]:
df["is_holiday"] = df["description"].notna().astype(int)


## Promotion cleanup

In [15]:
df["onpromotion"] = df["onpromotion"].fillna(False).astype(int)

C:\Users\argon\AppData\Local\Temp\ipykernel_15760\4182541060.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["onpromotion"] = df["onpromotion"].fillna(False).astype(int)


## Lag features

In [16]:
LAGS = [7, 14, 28]

for lag in LAGS:
    df[f"lag_{lag}"] = (
        df.groupby(["store_nbr", "item_nbr"])["unit_sales"]
          .shift(lag)
    )


## Rolling features

In [17]:
ROLLS = [7, 14]

for window in ROLLS:
    df[f"rolling_{window}"] = (
        df.groupby(["store_nbr", "item_nbr"])["unit_sales"]
          .shift(1)
          .rolling(window)
          .mean()
    )


## Handle missing values from lags

In [18]:
lag_cols = [f"lag_{l}" for l in LAGS] + [f"rolling_{r}" for r in ROLLS]

df_model = df.dropna(subset=lag_cols).reset_index(drop=True)

## Select model features

In [19]:
FEATURES = [
    # identifiers
    "store_nbr", "item_nbr",

    # promotions & ops
    "onpromotion", "transactions", "perishable",

    # calendar
    "month", "dayofweek", "weekofyear", "is_weekend",

    # external
    "dcoilwtico", "is_holiday",

    # time-series
    "lag_7", "lag_14", "lag_28",
    "rolling_7", "rolling_14"
]

TARGET = "unit_sales"


## Final sanity check

In [20]:
# Check for missing values
df_model[FEATURES + [TARGET]].isna().mean().sort_values(ascending=False)


dcoilwtico      0.307635
store_nbr       0.000000
rolling_14      0.000000
rolling_7       0.000000
lag_28          0.000000
lag_14          0.000000
lag_7           0.000000
is_holiday      0.000000
is_weekend      0.000000
item_nbr        0.000000
weekofyear      0.000000
dayofweek       0.000000
month           0.000000
perishable      0.000000
transactions    0.000000
onpromotion     0.000000
unit_sales      0.000000
dtype: float64

In [21]:
df_model = df_model.sort_values("date")

df_model["dcoilwtico"] = (
    df_model["dcoilwtico"]
    .ffill()
    .bfill()   # safety for very early dates
)


In [22]:
df_model["dcoilwtico"].isna().mean()


0.0

In [26]:
df_model["transactions"] = df_model["transactions"].fillna(0)


In [23]:
df_model["transactions"].isna().mean()


0.0

In [24]:
df_model[FEATURES + [TARGET]].isna().mean()


store_nbr       0.0
item_nbr        0.0
onpromotion     0.0
transactions    0.0
perishable      0.0
month           0.0
dayofweek       0.0
weekofyear      0.0
is_weekend      0.0
dcoilwtico      0.0
is_holiday      0.0
lag_7           0.0
lag_14          0.0
lag_28          0.0
rolling_7       0.0
rolling_14      0.0
unit_sales      0.0
dtype: float64

In [25]:
df_model.to_parquet(
    "favorita_model_ready_2013_2015.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False
)


In [26]:
df_model.groupby(df_model["date"].dt.year).size()


date
2013    3582223
2014    4934336
2015    5396951
dtype: int64

In [27]:
df_model.groupby(df_model["date"].dt.year)["item_nbr"].nunique()


date
2013    588
2014    766
2015    792
Name: item_nbr, dtype: int64

# Deployment Test Set

In [28]:
train = pd.read_csv(
    r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\train.csv\train.csv",
    parse_dates=["date"]
)

stores = pd.read_csv(
    r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\stores.csv\stores.csv"
)

items = pd.read_csv(
    r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\items.csv\items.csv"
)

oil = pd.read_csv(
    r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\oil.csv\oil.csv",
    parse_dates=["date"]
)

holidays = pd.read_csv(
    r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\holidays_events.csv\holidays_events.csv",
    parse_dates=["date"]
)

transactions = pd.read_csv(
    r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\transactions.csv\transactions.csv",
    parse_dates=["date"]
)


C:\Users\argon\AppData\Local\Temp\ipykernel_15760\3834988889.py:1: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(


## Recompute store & item filters (same logic as training)

In [29]:
store_volume = (
    train.groupby("store_nbr")["unit_sales"]
    .sum()
    .sort_values(ascending=False)
)

sample_stores = store_volume.head(25).index.tolist()

item_history = (
    train.groupby("item_nbr")
    .agg(
        total_sales=("unit_sales", "sum"),
        n_obs=("unit_sales", "count")
    )
)

sample_items = (
    item_history
    .query("n_obs > 500")
    .sort_values("total_sales", ascending=False)
    .head(800)
    .index
    .tolist()
)


## Filter to required dataset

In [30]:
df = train[
    train["store_nbr"].isin(sample_stores) &
    train["item_nbr"].isin(sample_items)
].copy()

df = df[
    (df["date"] >= "2016-01-01") &
    (df["date"] <= "2016-03-31")
]


## Apply joins

In [31]:
df = df.merge(items, on="item_nbr", how="left")
df = df.merge(stores, on="store_nbr", how="left")
df = df.merge(oil, on="date", how="left")

df = df.merge(
    holidays[["date", "description"]],
    on="date",
    how="left"
)

df = df.merge(
    transactions,
    on=["date", "store_nbr"],
    how="left"
)


## Calendar and Flag features

In [32]:
df = df.sort_values(["store_nbr", "item_nbr", "date"]).reset_index(drop=True)

df["month"] = df["date"].dt.month
df["dayofweek"] = df["date"].dt.dayofweek
df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

df["is_holiday"] = df["description"].notna().astype(int)
df["onpromotion"] = df["onpromotion"].fillna(False).astype(int)


C:\Users\argon\AppData\Local\Temp\ipykernel_15760\4260061521.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["onpromotion"] = df["onpromotion"].fillna(False).astype(int)


## Lag and Rolling Features

In [34]:
LAGS = [7, 14, 28]
ROLLS = [7, 14]

for lag in LAGS:
    df[f"lag_{lag}"] = (
        df.groupby(["store_nbr", "item_nbr"])["unit_sales"]
          .shift(lag)
    )

for window in ROLLS:
    df[f"rolling_{window}"] = (
        df.groupby(["store_nbr", "item_nbr"])["unit_sales"]
          .shift(1)
          .rolling(window)
          .mean()
    )


## Drop invalid rows

In [36]:
lag_cols = [f"lag_{l}" for l in LAGS] + [f"rolling_{r}" for r in ROLLS]

df_model = df.dropna(subset=lag_cols).reset_index(drop=True)


## Final cleaning

In [37]:
df_model["dcoilwtico"] = (
    df_model["dcoilwtico"]
    .ffill()
    .bfill()
)

df_model["transactions"] = df_model["transactions"].fillna(0)


## Extract deployment set

In [39]:
deployment_test_df = df_model[
    (df_model["date"] >= "2016-02-01") &
    (df_model["date"] <= "2016-03-31")
].copy()

print("Deployment test shape:", deployment_test_df.shape)


Deployment test shape: (940950, 26)


In [40]:
deployment_test_df.to_parquet(
    "favorita_deployment_test_2016_02_03.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False
)


In [41]:
deployment_test_df.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,...,month,dayofweek,weekofyear,is_weekend,is_holiday,lag_7,lag_14,lag_28,rolling_7,rolling_14
0,69383002,2016-02-02,2,105574,1.0,0,GROCERY I,1045,0,Quito,...,2,1,5,0,0,2.0,1.0,1.0,2.571429,2.642857
1,69478613,2016-02-03,2,105574,1.0,0,GROCERY I,1045,0,Quito,...,2,2,5,0,0,6.0,6.0,1.0,2.428571,2.642857
2,69760855,2016-02-06,2,105574,3.0,0,GROCERY I,1045,0,Quito,...,2,5,5,1,0,1.0,5.0,10.0,1.714286,2.285714
3,69860548,2016-02-07,2,105574,4.0,0,GROCERY I,1045,0,Quito,...,2,6,5,1,0,1.0,1.0,5.0,2.000000,2.142857
4,69952246,2016-02-08,2,105574,3.0,0,GROCERY I,1045,0,Quito,...,2,0,6,0,1,2.0,3.0,6.0,2.428571,2.357143


In [46]:
# Pick a random (date, store) that actually exists
date_store = (
    deployment_test_df[["date", "store_nbr"]]
    .drop_duplicates()
    .sample(1)
    .iloc[0]
)

date = date_store["date"]
store = date_store["store_nbr"]

date, store


(Timestamp('2016-02-16 00:00:00'), 45)

In [56]:
batch_items = (
    deployment_test_df[
        (deployment_test_df["date"] == date) &
        (deployment_test_df["store_nbr"] == store)
    ]
    .sample(40)   # batch size
    [["date", "store_nbr", "item_nbr", "onpromotion"]]
)

batch_items


,date,store_nbr,item_nbr,onpromotion
707201,2016-02-16,45,1464218,0
680232,2016-02-16,45,426155,0
701448,2016-02-16,45,1239905,0
694457,2016-02-16,45,1047685,0
694169,2016-02-16,45,1047675,0
697363,2016-02-16,45,1114567,0
710746,2016-02-16,45,1695828,0
683283,2016-02-16,45,564288,0
689814,2016-02-16,45,862454,0
706726,2016-02-16,45,1464070,0


In [49]:
sample = (
    deployment_test_df
    .dropna(subset=["store_nbr", "item_nbr"])
    .sample(5, random_state=42)
)

payload = {
    "date": sample.iloc[0]["date"].strftime("%Y-%m-%d"),
    "store_nbr": int(sample.iloc[0]["store_nbr"]),
    "items": [
        {
            "item_nbr": int(row["item_nbr"]),
            "onpromotion": bool(row["onpromotion"])
        }
        for _, row in sample.iterrows()
    ]
}

payload


{'date': '2016-03-04',
 'store_nbr': 37,
 'items': [{'item_nbr': 769314, 'onpromotion': True},
  {'item_nbr': 1696007, 'onpromotion': False},
  {'item_nbr': 108831, 'onpromotion': True},
  {'item_nbr': 315220, 'onpromotion': False},
  {'item_nbr': 352513, 'onpromotion': False}]}

In [50]:
train

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN
...,...,...,...,...,...,...
125497035,125497035,2017-08-15,54,2089339,4.0,False
125497036,125497036,2017-08-15,54,2106464,1.0,True
125497037,125497037,2017-08-15,54,2110456,192.0,False
125497038,125497038,2017-08-15,54,2113914,198.0,True


In [52]:
test = pd.read_csv(r"C:\Users\argon\Downloads\favorita-grocery-sales-forecasting\test.csv\test.csv")
test.tail(10)

,id,date,store_nbr,item_nbr,onpromotion
3370454,128867494,2017-08-31,54,2130526,False
3370455,128867495,2017-08-31,54,2130553,False
3370456,128867496,2017-08-31,54,2131010,False
3370457,128867497,2017-08-31,54,2131572,False
3370458,128867498,2017-08-31,54,2131699,False
3370459,128867499,2017-08-31,54,2132163,False
3370460,128867500,2017-08-31,54,2132318,False
3370461,128867501,2017-08-31,54,2132945,False
3370462,128867502,2017-08-31,54,2132957,False
3370463,128867503,2017-08-31,54,2134244,False


In [53]:
len(test)

3370464

In [54]:
train.tail(1)

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
125497039,125497039,2017-08-15,54,2116416,2.0,False
